In [1]:
import pandas as pd
import numpy as np
import csv
import matplotlib.pyplot as plt
from decimal import *
import re
from scipy.interpolate import interp1d

#### Estimating the population of Manaus between 2000, 2010, 2022 and 2025 by linear interpolation

In [3]:
pop_manaus_2000 = 1403796 ### https://biblioteca.ibge.gov.br/index.php/biblioteca-catalogo?view=detalhes&id=7308 (Tabelas Excel > AM > UF > Tabela 15)
pop_manaus_2010 = 1802014 ### https://www.ibge.gov.br/estatisticas/sociais/populacao/9662-censo-demografico-2010.html?=&t=resultados (Amazonas > Tabela 2.1.3)
pop_manaus_2022 = 2063689 ### https://cidades.ibge.gov.br/brasil/am/manaus/panorama, https://www.ibge.gov.br/cidades-e-estados/am/manaus.html
pop_manaus_2025 = 2303732 ### ESTIMATED, https://cidades.ibge.gov.br/brasil/am/manaus/panorama, https://www.ibge.gov.br/cidades-e-estados/am/manaus.html


ibge_pop_data = {
    'YEAR': [2000, 2010, 2022, 2025],
    'POPULATION': [pop_manaus_2000, pop_manaus_2010, pop_manaus_2022, pop_manaus_2025]
}

ibge_pop_data_df = pd.DataFrame(ibge_pop_data)

ibge_pop_data_df.set_index('YEAR', inplace=True)

ibge_pop_data_df_interpolated = ibge_pop_data_df.reindex(range(2000, 2026))  # Reindex to include all years from 2000 to 2025
ibge_pop_data_df_interpolated['POPULATION'] = ibge_pop_data_df_interpolated['POPULATION'].interpolate(method='linear')

ibge_pop_data_df_interpolated.reset_index(inplace=True)

ibge_pop_data_df_interpolated['POPULATION'] = ibge_pop_data_df_interpolated['POPULATION'].astype(int)

ibge_pop_data_df_interpolated['MUNIC_RES'] = '130260'
ibge_pop_data_df_interpolated

,YEAR,POPULATION,MUNIC_RES
0,2000,1403796,130260
1,2001,1443617,130260
2,2002,1483439,130260
3,2003,1523261,130260
4,2004,1563083,130260
5,2005,1602905,130260
6,2006,1642726,130260
7,2007,1682548,130260
8,2008,1722370,130260
9,2009,1762192,130260


In [4]:
ibge_pop_data_df_2016_2024 = ibge_pop_data_df_interpolated[(ibge_pop_data_df_interpolated.get('YEAR') >= 2016)
                                                         & (ibge_pop_data_df_interpolated.get('YEAR') <= 2024)].copy()
ibge_pop_data_df_2016_2024

,YEAR,POPULATION,MUNIC_RES
16,2016,1932851,130260
17,2017,1954657,130260
18,2018,1976464,130260
19,2019,1998270,130260
20,2020,2020076,130260
21,2021,2041882,130260
22,2022,2063689,130260
23,2023,2143703,130260
24,2024,2223717,130260


#### Estimated rural population of Manaus (~ 0.45% of total population)

In [6]:
ibge_rural_pop_data_df = ibge_pop_data_df_interpolated.copy()
ibge_rural_pop_data_df['RURAL_POP'] = ibge_rural_pop_data_df['POPULATION']*0.45/100
ibge_rural_pop_data_df

,YEAR,POPULATION,MUNIC_RES,RURAL_POP
0,2000,1403796,130260,6317.0820
1,2001,1443617,130260,6496.2765
2,2002,1483439,130260,6675.4755
3,2003,1523261,130260,6854.6745
4,2004,1563083,130260,7033.8735
5,2005,1602905,130260,7213.0725
6,2006,1642726,130260,7392.2670
7,2007,1682548,130260,7571.4660
8,2008,1722370,130260,7750.6650
9,2009,1762192,130260,7929.8640


In [7]:
ibge_rural_pop_data_df_2016_2025 = ibge_rural_pop_data_df[
    (ibge_rural_pop_data_df['YEAR'] >= 2016)# &
    #(ibge_rural_pop_data_df['YEAR'] <= 2024)
].copy()

ibge_rural_pop_data_df_2016_2025

,YEAR,POPULATION,MUNIC_RES,RURAL_POP
16,2016,1932851,130260,8697.8295
17,2017,1954657,130260,8795.9565
18,2018,1976464,130260,8894.0880
19,2019,1998270,130260,8992.2150
20,2020,2020076,130260,9090.3420
21,2021,2041882,130260,9188.4690
22,2022,2063689,130260,9286.6005
23,2023,2143703,130260,9646.6635
24,2024,2223717,130260,10006.7265
25,2025,2303732,130260,10366.7940


In [8]:
ibge_rural_pop_data_df_2016_2025['YEAR'] = ibge_rural_pop_data_df_2016_2025['YEAR'].astype(int)

# Create a date column (Jan 1 of each year)
ibge_rural_pop_data_df_2016_2025['DATE'] = pd.to_datetime(ibge_rural_pop_data_df_2016_2025['YEAR'], format='%Y')

# Set as index
ibge_rural_pop_data_df_2016_2025 = ibge_rural_pop_data_df_2016_2025.set_index('DATE')

# Create full daily range
daily_index = pd.date_range(start='2016-01-01', end='2025-12-31', freq='D')

# Reindex
ibge_rural_pop_data_df_2016_2025_daily = ibge_rural_pop_data_df_2016_2025.reindex(daily_index)

ibge_rural_pop_data_df_2016_2025_daily['ADJUSTED_POPULATION'] = ibge_rural_pop_data_df_2016_2025_daily['POPULATION'].interpolate(method='linear')
ibge_rural_pop_data_df_2016_2025_daily['ADJUSTED_RURAL_POP'] = ibge_rural_pop_data_df_2016_2025_daily['RURAL_POP'].interpolate(method='linear')

ibge_rural_pop_data_df_2016_2024_daily = ibge_rural_pop_data_df_2016_2025_daily[
    (ibge_rural_pop_data_df_2016_2025_daily.index.year <= 2024)
].copy()

clean_ibge_rural_pop_data_df_2016_2024 = ibge_rural_pop_data_df_2016_2024_daily.drop(columns=['YEAR', 'POPULATION', 'MUNIC_RES', 'RURAL_POP'])
clean_ibge_rural_pop_data_df_2016_2024

,ADJUSTED_POPULATION,ADJUSTED_RURAL_POP
2016-01-01,1.932851e+06,8697.829500
2016-01-02,1.932911e+06,8698.097607
2016-01-03,1.932970e+06,8698.365713
2016-01-04,1.933030e+06,8698.633820
2016-01-05,1.933089e+06,8698.901926
...,...,...
2024-12-27,2.302639e+06,10361.875045
2024-12-28,2.302858e+06,10362.858836
2024-12-29,2.303076e+06,10363.842627
2024-12-30,2.303295e+06,10364.826418


In [9]:
#### Mean daily growth of population from 2016 to 2024
start_date = clean_ibge_rural_pop_data_df_2016_2024.index.min()
end_date = clean_ibge_rural_pop_data_df_2016_2024.index.max()

start_pop = clean_ibge_rural_pop_data_df_2016_2024.loc[start_date, 'ADJUSTED_POPULATION']
end_pop = clean_ibge_rural_pop_data_df_2016_2024.loc[end_date, 'ADJUSTED_POPULATION']

mean_daily_growth = (end_pop - start_pop) / (end_date - start_date).days
mean_daily_growth ### np.round(mean_daily_growth)

# #### (Pop from 2024 - Pop from 2016)/Days between 2016 and 2024
# (
#     clean_ibge_rural_pop_data_df_2016_2024.loc[clean_ibge_rural_pop_data_df_2016_2024.index.year == 2024, 'ADJUSTED_POPULATION'].values[-1] - 
#     clean_ibge_rural_pop_data_df_2016_2024.loc[clean_ibge_rural_pop_data_df_2016_2024.index.year == 2016, 'ADJUSTED_POPULATION'].values[0]
# )/(3*366 + 6*365)

np.float64(112.76616360858559)

In [10]:
#### Mean daily growth of rural population from 2016 to 2024
start_date = clean_ibge_rural_pop_data_df_2016_2024.index.min()
end_date = clean_ibge_rural_pop_data_df_2016_2024.index.max()

start_rural_pop = clean_ibge_rural_pop_data_df_2016_2024.loc[start_date, 'ADJUSTED_RURAL_POP']
end_rural_pop = clean_ibge_rural_pop_data_df_2016_2024.loc[end_date, 'ADJUSTED_RURAL_POP']

mean_daily_growth = (end_rural_pop - start_rural_pop) / (end_date - start_date).days
mean_daily_growth

np.float64(0.5074477362386353)

In [11]:
ibge_rural_pop_data_df_2016_2023_daily = ibge_rural_pop_data_df_2016_2025_daily[
    (ibge_rural_pop_data_df_2016_2025_daily.index.year <= 2023)
].copy()

clean_ibge_rural_pop_data_df_2016_2023 = ibge_rural_pop_data_df_2016_2023_daily.drop(columns=['YEAR', 'POPULATION', 'MUNIC_RES', 'RURAL_POP'])

In [12]:
#### Mean daily growth of population from 2016 to 2023
start_date = clean_ibge_rural_pop_data_df_2016_2023.index.min()
end_date = clean_ibge_rural_pop_data_df_2016_2023.index.max()

start_pop = clean_ibge_rural_pop_data_df_2016_2023.loc[start_date, 'ADJUSTED_POPULATION']
end_pop = clean_ibge_rural_pop_data_df_2016_2023.loc[end_date, 'ADJUSTED_POPULATION']

mean_daily_growth = (end_pop - start_pop) / (end_date - start_date).days
mean_daily_growth

np.float64(99.50249351648192)

In [13]:
per_capita_growth = (
    clean_ibge_rural_pop_data_df_2016_2023['ADJUSTED_POPULATION']
    .diff()
    .div(clean_ibge_rural_pop_data_df_2016_2023['ADJUSTED_POPULATION'])
    .mean()
)

print(f"{per_capita_growth:.10f}")

0.0000479564


In [14]:
#### Mean daily growth of rural population from 2016 to 2023
start_date = clean_ibge_rural_pop_data_df_2016_2023.index.min()
end_date = clean_ibge_rural_pop_data_df_2016_2023.index.max()

start_rural_pop = clean_ibge_rural_pop_data_df_2016_2023.loc[start_date, 'ADJUSTED_RURAL_POP']
end_rural_pop = clean_ibge_rural_pop_data_df_2016_2023.loc[end_date, 'ADJUSTED_RURAL_POP']

mean_daily_growth = (end_rural_pop - start_rural_pop) / (end_date - start_date).days
mean_daily_growth

np.float64(0.44776122082416886)

In [15]:
per_capita_rural_growth = (
    clean_ibge_rural_pop_data_df_2016_2023['ADJUSTED_RURAL_POP']
    .diff()
    .div(clean_ibge_rural_pop_data_df_2016_2023['ADJUSTED_RURAL_POP'])
    .mean()
)

print(f"{per_capita_rural_growth:.10f}")

0.0000479564
